# Credit Risk — LightGBM Modeling

**Data source** (priority order):
1. `data/interim/loans_features.parquet` — feature-engineered output from `02_feature_engineering.ipynb`
2. `data/processed/loans_cleaned.parquet` — fallback (LightGBM handles raw categoricals natively)

**Target**: `default` (1 = Charged Off/Default/Late 31-120d, 0 = Fully Paid, ~21% positive rate)  
**Experiment tracking**: MLflow → `mlruns/`

---
### Table of Contents
1. [Setup & Load](#1)
2. [Feature Preparation](#2)
3. [Train / Val / Test Split](#3)
4. [LightGBM Baseline Model](#4)
5. [Evaluation — Val Set](#5)
6. [Threshold Optimisation](#6)
7. [Hyperparameter Tuning — Optuna](#7)
8. [Best Model — Test Set Evaluation](#8)
9. [SHAP Explainability](#9)
10. [Save Model & Artifacts](#10)

---
## 1. Setup & Load <a id="1"></a>

In [3]:
from pathlib import Path
import warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    f1_score, brier_score_loss,
)

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    # Optuna 4.x moved integrations to optuna-integration package
    try:
        from optuna_integration import LightGBMPruningCallback as _LGBMPruningCB
        OPTUNA_PRUNING = True
    except ImportError:
        OPTUNA_PRUNING = False
    OPTUNA = True
    print(f"Optuna {optuna.__version__} available — HPO enabled (pruning={'on' if OPTUNA_PRUNING else 'off, install optuna-integration[lightgbm]'})")
except ImportError:
    OPTUNA = False
    OPTUNA_PRUNING = False
    print("Optuna not installed — skipping HPO section (pip install optuna to enable)")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120})

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path("..").resolve()
INTERIM_PATH   = PROJECT_ROOT / "data" / "interim"   / "loans_features.parquet"
CLEANED_PATH   = PROJECT_ROOT / "data" / "processed" / "loans_cleaned.parquet"
MLFLOW_URI     = str(PROJECT_ROOT / "mlruns")
MODEL_DIR      = PROJECT_ROOT / "mlruns" / "artifacts"

RANDOM_STATE   = 42
TARGET         = "default"

print(f"LightGBM  : {lgb.__version__}")
print(f"MLflow    : {mlflow.__version__}")
print(f"SHAP      : {shap.__version__}")

Optuna 4.8.0 available — HPO enabled (pruning=off, install optuna-integration[lightgbm])
LightGBM  : 4.6.0
MLflow    : 3.11.1
SHAP      : 0.51.0


In [4]:
# ── Load data — prefer feature-engineered interim file ────────────────────
if INTERIM_PATH.exists():
    df = pd.read_parquet(INTERIM_PATH)
    DATA_SOURCE = "interim (feature-engineered)"
else:
    df = pd.read_parquet(CLEANED_PATH)
    DATA_SOURCE = "processed (raw cleaned — LightGBM native categoricals)"
    print("[WARNING] Interim features not found. Run 02_feature_engineering.ipynb first.")
    print("          Falling back to cleaned data — LightGBM will handle categoricals natively.")

print(f"Source    : {DATA_SOURCE}")
print(f"Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Default rate : {df[TARGET].mean()*100:.2f}%")

Source    : interim (feature-engineered)
Shape     : 1,369,566 rows × 100 columns
Default rate : 21.23%


---
## 2. Feature Preparation <a id="2"></a>

In [5]:
# ── Columns to always drop (leaky, redundant, or ID-like) ─────────────────
ALWAYS_DROP = [
    TARGET,               # target — separated below
    "funded_amnt",        # ≈ loan_amnt (collinear)
    "funded_amnt_inv",    # ≈ loan_amnt
    "fico_range_high",    # ≈ fico_range_low
    "emp_title",          # raw high-cardinality string — no numeric mapping
]
ALWAYS_DROP = [c for c in ALWAYS_DROP if c in df.columns]

feature_cols = [c for c in df.columns if c not in ALWAYS_DROP + [TARGET]]

X_raw = df[feature_cols].copy()
y     = df[TARGET].astype("int32")

# ── LightGBM native categorical support ───────────────────────────────────
# String/object columns → cast to pandas Categorical so lgb.Dataset can use them
cat_cols = X_raw.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    X_raw[col] = X_raw[col].astype("category")

num_cols = X_raw.select_dtypes(include="number").columns.tolist()

print(f"Feature columns : {len(feature_cols)}")
print(f"  Numeric       : {len(num_cols)}")
print(f"  Categorical   : {len(cat_cols)}  →  {cat_cols}")
print(f"  Target        : {y.value_counts().to_dict()}")

Feature columns : 99
  Numeric       : 99
  Categorical   : 0  →  []
  Target        : {0: 1078739, 1: 290827}


---
## 3. Train / Val / Test Split <a id="3"></a>

- **Train** 70% — model fits here  
- **Val**   10% — early stopping + threshold tuning  
- **Test**  20% — held-out final evaluation (touch once)

In [6]:
# Stratified split to preserve ~21% default rate in all three sets
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,        # 0.125 × 0.80 = 0.10 of full dataset
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print(f"Train : {len(X_train):>9,} rows  |  default rate: {y_train.mean()*100:.2f}%")
print(f"Val   : {len(X_val):>9,} rows  |  default rate: {y_val.mean()*100:.2f}%")
print(f"Test  : {len(X_test):>9,} rows  |  default rate: {y_test.mean()*100:.2f}%")

# Positive weight for scale_pos_weight (minority / majority)
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
SCALE_POS_WEIGHT = neg / pos
print(f"\nscale_pos_weight : {SCALE_POS_WEIGHT:.3f}  (neg={neg:,} / pos={pos:,})")

Train :   958,695 rows  |  default rate: 21.23%
Val   :   136,957 rows  |  default rate: 21.24%
Test  :   273,914 rows  |  default rate: 21.24%

scale_pos_weight : 3.709  (neg=755,117 / pos=203,578)


In [7]:
# Build lgb.Dataset objects (most efficient format for LightGBM)
lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=cat_cols if cat_cols else "auto",
    free_raw_data=False,
)
lgb_val = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=cat_cols if cat_cols else "auto",
    reference=lgb_train,
    free_raw_data=False,
)

print("lgb.Dataset objects created.")

lgb.Dataset objects created.


---
## 4. LightGBM Baseline Model <a id="4"></a>

Params sourced from `configs/model.yaml` → `models.lightgbm`.  
Early stopping on val AUC with patience = 50 rounds.

In [8]:
BASELINE_PARAMS = {
    # Core
    "objective"        : "binary",
    "metric"           : ["auc", "binary_logloss", "average_precision"],
    "boosting_type"    : "gbdt",
    "verbosity"        : -1,
    "random_state"     : RANDOM_STATE,
    # Tree structure
    "num_leaves"       : 63,
    "max_depth"        : 6,
    "min_child_samples": 50,
    # Regularisation
    "learning_rate"    : 0.05,
    "n_estimators"     : 1000,       # controlled by early stopping
    "subsample"        : 0.8,
    "subsample_freq"   : 1,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.0,
    # Class imbalance
    "scale_pos_weight" : SCALE_POS_WEIGHT,
    # Speed
    "n_jobs"           : -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

print("Baseline params:")
for k, v in BASELINE_PARAMS.items():
    print(f"  {k:<22} : {v}")

Baseline params:
  objective              : binary
  metric                 : ['auc', 'binary_logloss', 'average_precision']
  boosting_type          : gbdt
  verbosity              : -1
  random_state           : 42
  num_leaves             : 63
  max_depth              : 6
  min_child_samples      : 50
  learning_rate          : 0.05
  n_estimators           : 1000
  subsample              : 0.8
  subsample_freq         : 1
  colsample_bytree       : 0.8
  reg_alpha              : 0.1
  reg_lambda             : 1.0
  scale_pos_weight       : 3.709226930218393
  n_jobs                 : -1


In [9]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("loan-default-lightgbm")

with mlflow.start_run(run_name="baseline") as run:
    RUN_ID_BASELINE = run.info.run_id

    t0 = time.time()
    baseline_model = lgb.train(
        params          = BASELINE_PARAMS,
        train_set       = lgb_train,
        num_boost_round = BASELINE_PARAMS["n_estimators"],
        valid_sets      = [lgb_train, lgb_val],
        valid_names     = ["train", "val"],
        callbacks       = callbacks,
    )
    train_time = time.time() - t0

    # Predict probabilities
    val_proba  = baseline_model.predict(X_val,  num_iteration=baseline_model.best_iteration)
    train_proba = baseline_model.predict(X_train, num_iteration=baseline_model.best_iteration)

    # Core metrics
    val_auc  = roc_auc_score(y_val, val_proba)
    val_ap   = average_precision_score(y_val, val_proba)
    val_brier = brier_score_loss(y_val, val_proba)
    train_auc = roc_auc_score(y_train, train_proba)

    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_params({"best_iteration": baseline_model.best_iteration,
                       "feature_count": len(feature_cols),
                       "data_source": DATA_SOURCE})
    mlflow.log_metrics({
        "val_roc_auc"          : val_auc,
        "val_avg_precision"    : val_ap,
        "val_brier_score"      : val_brier,
        "train_roc_auc"        : train_auc,
        "train_time_seconds"   : train_time,
    })
    mlflow.lightgbm.log_model(baseline_model, artifact_path="baseline_model")

print(f"\n{'='*50}")
print(f"  Best iteration : {baseline_model.best_iteration}")
print(f"  Train ROC-AUC  : {train_auc:.4f}")
print(f"  Val   ROC-AUC  : {val_auc:.4f}")
print(f"  Val   PR-AUC   : {val_ap:.4f}")
print(f"  Val   Brier    : {val_brier:.4f}")
print(f"  Train time     : {train_time:.1f}s")
print(f"  MLflow run     : {RUN_ID_BASELINE}")
print(f"{'='*50}")

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlflow/store/tracking/file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlflow/store/tracking/file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlflow/utils/yaml_util

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	train's auc: 0.718898	train's binary_logloss: 0.502715	train's average_precision: 0.401564	val's auc: 0.718615	val's binary_logloss: 0.502747	val's average_precision: 0.40014


2026/05/03 12:57:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  Best iteration : 5
  Train ROC-AUC  : 0.7189
  Val   ROC-AUC  : 0.7186
  Val   PR-AUC   : 0.4001
  Val   Brier    : 0.1617
  Train time     : 6.0s
  MLflow run     : a4697c54c47b424c936e7a338d591573


---
## 5. Evaluation — Val Set <a id="5"></a>

In [ ]:
# ── ROC Curve ──────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_val, val_proba)
prec, rec, _  = precision_recall_curve(y_val, val_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, color="#5C6BC0", lw=2,
             label=f"LightGBM (AUC = {val_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Val Set")
axes[0].legend()

# PR Curve
axes[1].plot(rec, prec, color="#E53935", lw=2,
             label=f"LightGBM (AP = {val_ap:.4f})")
baseline_precision = y_val.mean()
axes[1].axhline(baseline_precision, ls="--", color="grey", lw=1,
                label=f"Random ({baseline_precision:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Val Set")
axes[1].legend()

plt.suptitle("LightGBM Baseline — Validation Performance", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Probability calibration check ─────────────────────────────────────────
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 5))
prob_true, prob_pred = calibration_curve(y_val, val_proba, n_bins=20, strategy="quantile")
ax.plot(prob_pred, prob_true, marker="o", color="#5C6BC0", lw=2, label="LightGBM")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curve — Val Set")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted probability distribution by class ───────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
for label, color, name in [(0, "#1E88E5", "Fully Paid"), (1, "#E53935", "Default")]:
    mask = y_val == label
    ax.hist(val_proba[mask], bins=80, alpha=0.55, color=color,
            label=f"{name} (n={mask.sum():,})", density=True)

ax.set_xlabel("Predicted Default Probability")
ax.set_ylabel("Density")
ax.set_title("Score Distribution by True Class")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Threshold Optimisation <a id="6"></a>

Default 0.5 threshold often sub-optimal for imbalanced credit risk.  
We sweep thresholds on the **val set** and pick the one maximising F1.

In [ ]:
thresholds  = np.linspace(0.01, 0.99, 200)
f1_scores   = [f1_score(y_val, (val_proba >= t).astype(int), zero_division=0) for t in thresholds]

best_idx    = int(np.argmax(f1_scores))
BEST_THRESH = thresholds[best_idx]
BEST_F1_VAL = f1_scores[best_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores, color="#5C6BC0", lw=2)
ax.axvline(BEST_THRESH, color="#E53935", ls="--", lw=1.5,
           label=f"Best threshold = {BEST_THRESH:.3f}  (F1 = {BEST_F1_VAL:.4f})")
ax.axvline(0.5, color="grey", ls=":", lw=1.2, label="Default 0.50")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("F1 vs Threshold — Val Set")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal threshold : {BEST_THRESH:.4f}")
print(f"Val F1 @ optimal  : {BEST_F1_VAL:.4f}")
print(f"Val F1 @ 0.50     : {f1_score(y_val, (val_proba >= 0.50).astype(int)):.4f}")

In [ ]:
# Confusion matrix at optimal threshold
val_pred = (val_proba >= BEST_THRESH).astype(int)
cm        = confusion_matrix(y_val, val_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt=",d", cmap="Blues",
    xticklabels=["Pred: Paid", "Pred: Default"],
    yticklabels=["True: Paid", "True: Default"],
    ax=ax,
)
ax.set_title(f"Confusion Matrix — Val Set  (thresh = {BEST_THRESH:.3f})")
plt.tight_layout()
plt.show()

print(classification_report(y_val, val_pred, target_names=["Fully Paid", "Default"]))

---
## 7. Hyperparameter Tuning — Optuna <a id="7"></a>

Objective: **maximise val ROC-AUC** within a 50-trial budget.  
*(Skipped if Optuna is not installed — install with `pip install optuna`)*

In [ ]:
if OPTUNA:
    def objective(trial):
        params = {
            "objective"         : "binary",
            "metric"            : "auc",
            "verbosity"         : -1,
            "boosting_type"     : "gbdt",
            "random_state"      : RANDOM_STATE,
            "n_jobs"            : -1,
            # Search space
            "num_leaves"        : trial.suggest_int("num_leaves", 31, 255),
            "max_depth"         : trial.suggest_int("max_depth", 4, 10),
            "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_samples" : trial.suggest_int("min_child_samples", 20, 200),
            "subsample"         : trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq"    : 1,
            "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "scale_pos_weight"  : SCALE_POS_WEIGHT,
        }

        callbacks = [
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1),
            ]
        if OPTUNA_PRUNING:
            callbacks.append(_LGBMPruningCB(trial, "auc", valid_name="val"))
        model = lgb.train(
            params          = params,
            train_set       = lgb_train,
            num_boost_round = 800,
            valid_sets      = [lgb_val],
            valid_names     = ["val"],
            callbacks       = callbacks,
        )
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        return roc_auc_score(y_val, preds)

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)
    study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    print("Running Optuna — 50 trials...")
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    BEST_PARAMS = {**BASELINE_PARAMS, **study.best_params}
    print(f"\nBest trial  : #{study.best_trial.number}")
    print(f"Best val AUC: {study.best_value:.4f}")
    print("Best params:")
    for k, v in study.best_params.items():
        print(f"  {k:<22} : {v}")
else:
    BEST_PARAMS = BASELINE_PARAMS.copy()
    print("Optuna not available — using baseline params as best params.")

In [ ]:
if OPTUNA:
    # Optimisation history plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    trial_nums = [t.number for t in study.trials if t.value is not None]
    trial_vals = [t.value  for t in study.trials if t.value is not None]
    best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]

    axes[0].scatter(trial_nums, trial_vals, alpha=0.5, s=20, color="#90CAF9")
    axes[0].plot(trial_nums, best_so_far, color="#E53935", lw=2, label="Best so far")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("Val ROC-AUC")
    axes[0].set_title("Optuna — Optimisation History")
    axes[0].legend()

    # Param importance
    importances = optuna.importance.get_param_importances(study)
    names = list(importances.keys())[:10]
    vals  = [importances[n] for n in names]
    axes[1].barh(names[::-1], vals[::-1], color="#5C6BC0")
    axes[1].set_xlabel("Importance")
    axes[1].set_title("Optuna — Hyperparameter Importance")

    plt.tight_layout()
    plt.show()

---
## 8. Best Model — Test Set Evaluation <a id="8"></a>

Retrain on **Train + Val** combined with the best params and early stopping disabled  
(we use the best iteration found during tuning).

In [ ]:
# Combine train + val for final refit
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

lgb_trainval = lgb.Dataset(
    X_trainval, label=y_trainval,
    categorical_feature=cat_cols if cat_cols else "auto",
    free_raw_data=False,
)
lgb_test_ds = lgb.Dataset(
    X_test, label=y_test,
    categorical_feature=cat_cols if cat_cols else "auto",
    reference=lgb_trainval,
    free_raw_data=False,
)

# Use best iteration from baseline (or tuned model if Optuna ran)
N_BEST_ITER = baseline_model.best_iteration

final_params = {k: v for k, v in BEST_PARAMS.items() if k != "n_estimators"}
final_params["verbosity"] = -1

with mlflow.start_run(run_name="final_model") as run:
    RUN_ID_FINAL = run.info.run_id

    t0 = time.time()
    final_model = lgb.train(
        params          = final_params,
        train_set       = lgb_trainval,
        num_boost_round = N_BEST_ITER,
        valid_sets      = [lgb_trainval, lgb_test_ds],
        valid_names     = ["trainval", "test"],
        callbacks       = [lgb.log_evaluation(period=N_BEST_ITER)],  # print once
    )
    train_time = time.time() - t0

    test_proba  = final_model.predict(X_test)
    test_pred   = (test_proba >= BEST_THRESH).astype(int)

    test_auc    = roc_auc_score(y_test, test_proba)
    test_ap     = average_precision_score(y_test, test_proba)
    test_f1     = f1_score(y_test, test_pred)
    test_brier  = brier_score_loss(y_test, test_proba)

    mlflow.log_params({**final_params, "num_boost_round": N_BEST_ITER,
                       "threshold": BEST_THRESH, "data_source": DATA_SOURCE})
    mlflow.log_metrics({
        "test_roc_auc"       : test_auc,
        "test_avg_precision" : test_ap,
        "test_f1"            : test_f1,
        "test_brier"         : test_brier,
        "val_roc_auc"        : val_auc,
        "val_avg_precision"  : val_ap,
        "train_time_seconds" : train_time,
    })
    mlflow.lightgbm.log_model(final_model, artifact_path="final_model")

print(f"\n{'='*55}")
print(f"  FINAL MODEL — TEST SET RESULTS")
print(f"{'='*55}")
print(f"  ROC-AUC          : {test_auc:.4f}")
print(f"  Avg Precision    : {test_ap:.4f}")
print(f"  F1 (thresh={BEST_THRESH:.2f}) : {test_f1:.4f}")
print(f"  Brier Score      : {test_brier:.4f}")
print(f"  MLflow run       : {RUN_ID_FINAL}")
print(f"{'='*55}")

In [ ]:
# ── Full evaluation plots — Test Set ──────────────────────────────────────
fpr_t, tpr_t, _  = roc_curve(y_test, test_proba)
prec_t, rec_t, _ = precision_recall_curve(y_test, test_proba)
cm_test           = confusion_matrix(y_test, test_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
axes[0].plot(fpr_t, tpr_t, color="#5C6BC0", lw=2,
             label=f"LightGBM (AUC = {test_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Random")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Test Set"); axes[0].legend()

# PR Curve
axes[1].plot(rec_t, prec_t, color="#E53935", lw=2,
             label=f"LightGBM (AP = {test_ap:.4f})")
axes[1].axhline(y_test.mean(), ls="--", color="grey", lw=1,
                label=f"Random ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR Curve — Test Set"); axes[1].legend()

# Confusion matrix
sns.heatmap(
    cm_test, annot=True, fmt=",d", cmap="Blues",
    xticklabels=["Pred: Paid", "Pred: Default"],
    yticklabels=["True: Paid", "True: Default"],
    ax=axes[2],
)
axes[2].set_title(f"Confusion Matrix  (thresh={BEST_THRESH:.3f})")

plt.suptitle("LightGBM Final Model — Test Set Evaluation", fontsize=13)
plt.tight_layout()
plt.show()

print(classification_report(y_test, test_pred, target_names=["Fully Paid", "Default"]))

In [ ]:
# ── Precision, Recall, F1 vs Threshold (test set) ─────────────────────────
from sklearn.metrics import precision_score, recall_score

thresholds_t  = np.linspace(0.01, 0.99, 200)
precisions_t  = [precision_score(y_test, (test_proba >= t).astype(int), zero_division=0) for t in thresholds_t]
recalls_t     = [recall_score(y_test,    (test_proba >= t).astype(int), zero_division=0) for t in thresholds_t]
f1s_t         = [f1_score(y_test,        (test_proba >= t).astype(int), zero_division=0) for t in thresholds_t]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds_t, precisions_t, label="Precision", color="#1E88E5", lw=2)
ax.plot(thresholds_t, recalls_t,    label="Recall",    color="#43A047", lw=2)
ax.plot(thresholds_t, f1s_t,        label="F1",        color="#E53935", lw=2)
ax.axvline(BEST_THRESH, ls="--", color="grey", lw=1.2, label=f"Chosen threshold = {BEST_THRESH:.3f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs Threshold — Test Set")
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. SHAP Explainability <a id="9"></a>

In [ ]:
# Sample for SHAP (full test set can be slow — 5k is enough for summary plots)
SHAP_SAMPLE = min(5_000, len(X_test))
X_shap = X_test.sample(SHAP_SAMPLE, random_state=RANDOM_STATE)

explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values shape : {np.array(shap_values).shape}")
print(f"Sample size       : {SHAP_SAMPLE:,}")

In [ ]:
# ── SHAP Summary — Beeswarm ───────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    max_display=20,
    show=False,
)
plt.title("SHAP Summary Plot — Top 20 Features", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Bar — Mean |SHAP| importance ─────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    plot_type="bar",
    max_display=20,
    show=False,
)
plt.title("SHAP Feature Importance — Mean |SHAP Value|", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Dependence plots — top 4 features ────────────────────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features  = X_shap.columns[np.argsort(mean_abs_shap)[::-1][:4]].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, feat_name in zip(axes, top_features):
    shap.dependence_plot(
        feat_name, shap_values, X_shap,
        ax=ax, show=False,
    )
    ax.set_title(f"SHAP Dependence: {feat_name}", fontsize=11)

plt.suptitle("SHAP Dependence Plots — Top 4 Features", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Waterfall plot — single high-risk prediction ───────────────────────────
# Find a true positive (default predicted as default) for illustration
shap_proba = final_model.predict(X_shap)
y_shap     = y_test.loc[X_shap.index]

tp_mask  = (y_shap == 1) & (shap_proba >= BEST_THRESH)
if tp_mask.any():
    example_idx = int(tp_mask[tp_mask].index[0])
    row_pos     = X_shap.index.get_loc(example_idx)

    shap_exp = shap.Explanation(
        values        = shap_values[row_pos],
        base_values   = explainer.expected_value,
        data          = X_shap.iloc[row_pos].values,
        feature_names = X_shap.columns.tolist(),
    )
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f"SHAP Waterfall — True Positive Example  (p={shap_proba[row_pos]:.3f})", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No true positive in SHAP sample — skip waterfall.")

---
## 10. Save Model & Artifacts <a id="10"></a>

In [ ]:
import joblib

SAVE_DIR = PROJECT_ROOT / "mlruns" / "artifacts"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── LightGBM native format ────────────────────────────────────────────────
model_path = SAVE_DIR / "lightgbm_final.txt"
final_model.save_model(str(model_path))

# ── Metadata ──────────────────────────────────────────────────────────────
metadata = {
    "model_type"       : "LightGBM",
    "mlflow_run_id"    : RUN_ID_FINAL,
    "best_iteration"   : N_BEST_ITER,
    "threshold"        : float(BEST_THRESH),
    "feature_columns"  : feature_cols,
    "categorical_cols" : cat_cols,
    "data_source"      : DATA_SOURCE,
    "test_roc_auc"     : float(test_auc),
    "test_avg_precision": float(test_ap),
    "test_f1"          : float(test_f1),
    "test_brier"       : float(test_brier),
    "params"           : final_params,
}
meta_path = SAVE_DIR / "lightgbm_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved  : {model_path}")
print(f"Metadata     : {meta_path}")
print(f"\nTo reload:")
print(f"  model = lgb.Booster(model_file='{model_path}')")
print(f"  or via MLflow: mlflow.lightgbm.load_model('runs:/{RUN_ID_FINAL}/final_model')")

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print("=" * 58)
print("  LIGHTGBM MODELING SUMMARY")
print("=" * 58)
print(f"  Data source       : {DATA_SOURCE}")
print(f"  Features used     : {len(feature_cols)}")
print(f"  Categorical cols  : {len(cat_cols)}")
print(f"  Train size        : {len(X_train):,}")
print(f"  Val size          : {len(X_val):,}")
print(f"  Test size         : {len(X_test):,}")
print()
print(f"  Best iteration    : {N_BEST_ITER}")
print(f"  Optimal threshold : {BEST_THRESH:.4f}")
print()
print("  --- Val Set ---")
print(f"  ROC-AUC           : {val_auc:.4f}")
print(f"  Avg Precision     : {val_ap:.4f}")
print(f"  F1                : {BEST_F1_VAL:.4f}")
print()
print("  --- Test Set ---")
print(f"  ROC-AUC           : {test_auc:.4f}")
print(f"  Avg Precision     : {test_ap:.4f}")
print(f"  F1                : {test_f1:.4f}")
print(f"  Brier Score       : {test_brier:.4f}")
print()
print(f"  MLflow baseline   : {RUN_ID_BASELINE}")
print(f"  MLflow final      : {RUN_ID_FINAL}")
print("=" * 58)